# Irradianza di una sorgente puntiforme
Si misura la tensione V del fotodiodo in funzione della distanza r dalla lampadina e si verifica la legge dell'inverso del quadrato.

- **Fit 1:** $V = A/(r-B)^2 + C$
- **Fit 2:** $V = A/(r-B)^D + C$ (esponente $D$ libero)

**Come si usa:** cambia solo le righe con scritto `CAMBIARE QUI` (cella *Parametri*), poi esegui le celle in ordine.

## Librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

!pip install -q iminuit
from iminuit import Minuit

## Parametri
Questa è l'**unica cella da modificare**.

Il file CSV deve avere (anche insieme ad altre) le colonne `x [mm]`, `V_dir [V]`, `V_fondo1 [V]`, `V_fondo2 [V]`, `V_fondo3 [V]`.
Se hai fatto una sola misura di fondo lascia vuote `V_fondo2` e `V_fondo3`.

In [ ]:
# --- File dati ---------------------------------------------------------------
nome_file = "dati_amici.csv"   # CAMBIARE QUI: nome del file csv

# --- Geometria:  r = (x - x0) + D/2 + profondità fotodiodo -----------------------
x0 = 122.0          # CAMBIARE QUI: lettura di x con il fotodiodo a contatto con la lampadina (mm)
D_lamp = 16.483     # CAMBIARE QUI: diametro lampadina (mm)
prof_diodo = 1.0    # profondità del fotodiodo nella scatoletta (mm)
r_min = 0.0         # CAMBIARE QUI (facoltativo): usa solo i punti con r >= r_min (mm); 0 = usa tutti

# --- Errori di misura (punto per punto) --------------------------------------------
dx = 0.2            # CAMBIARE QUI: errore di lettura su x (mm)
dV_dir = 0.01       # CAMBIARE QUI: sensibilità voltmetro su V_dir (V)
dV_fondo = 0.01     # CAMBIARE QUI: sensibilità voltmetro su V_fondo (V)
sigma_x = dx / np.sqrt(3)   # errore uniforme di semi-ampiezza dx  ->  σ = dx/√3

# --- Errori sistematici (comuni a tutti i punti: spostano tutte le r insieme) -------
# Servono solo per decidere alla fine se B è compatibile con 0. Sono già σ (non Δ).
sigma_x0 = 0.144    # mm
sigma_D = 0.05      # mm (errore sul diametro)
sigma_prof = 0.5    # mm (stima, da controllare)
sigma_sist = np.sqrt(sigma_x0**2 + (sigma_D / 2)**2 + sigma_prof**2)

## Dati
Dalle misure grezze si calcolano: tensione netta $V = V_{dir} - V_{fondo}$, distanza vera $r$ e i relativi errori.

In [ ]:
df = pd.read_csv(nome_file, sep=";", decimal=",", encoding="utf-8-sig")
df = df.dropna(subset=["x [mm]", "V_dir [V]", "V_fondo1 [V]"])   # scarta le righe vuote

pos = df["x [mm]"].to_numpy(dtype=float)       # lettura di x (mm)
V_dir = df["V_dir [V]"].to_numpy(dtype=float)  # tensione con lampada accesa (V)
fondo = df[["V_fondo1 [V]", "V_fondo2 [V]", "V_fondo3 [V]"]]   # celle vuote = NaN

# Tensione di fondo: media delle misure fatte e suo errore
V_fondo = fondo.mean(axis=1).to_numpy()
semidisp = ((fondo.max(axis=1) - fondo.min(axis=1)) / 2).to_numpy()
# con una sola misura (o misure tutte uguali) si usa la sensibilità del voltmetro
sigma_fondo = np.where(semidisp > 0, semidisp / np.sqrt(3), dV_fondo / np.sqrt(3))

# Grandezze usate nei fit
y = V_dir - V_fondo                                        # V netta (V)
ey = np.sqrt((dV_dir / np.sqrt(3))**2 + sigma_fondo**2)    # errore su V (V)
r = (pos - x0) + D_lamp / 2 + prof_diodo                   # distanza vera (mm)
ex = np.full_like(r, sigma_x)                              # errore su r (mm)

usa = r >= r_min                                           # taglio opzionale sulle distanze piccole
y, ey, r, ex = y[usa], ey[usa], r[usa], ex[usa]

print("numero di punti:", len(y))
print(f"r da {r.min():.1f} a {r.max():.1f} mm")

## Fit 1
Gli errori su r vengono portati su V con la derivata (varianza efficace):
$\sigma_{eff}^2 = \sigma_V^2 + \left(\sigma_r \, \frac{dV}{dr}\right)^2$

In [ ]:
def derivata(f, r, *par):
    # derivata numerica di f rispetto a r
    dr = 1e-6
    return (f(r + dr/2, *par) - f(r - dr/2, *par)) / dr

def compatibile(valore, atteso, errore):
    # stampa se valore e atteso sono compatibili entro 2 errori
    t = abs(valore - atteso) / errore
    esito = "compatibile" if t < 2 else "NON compatibile"
    print(f"   |valore - atteso| / errore = {t:.2f}  ->  {esito} (soglia 2)")

def fitfunc1(r, A, B, C):
    return A / (r - B)**2 + C

def chi2_1(A, B, C):
    sigma = np.sqrt(ey**2 + (ex * derivata(fitfunc1, r, A, B, C))**2)
    return np.sum(((y - fitfunc1(r, A, B, C)) / sigma)**2)

# parametri iniziali: A dal primo punto, B e C a zero
m1 = Minuit(chi2_1, A=y[0]*r[0]**2, B=0.0, C=0.0)
m1.errordef = 1
m1.migrad()
m1.hesse()

A1, B1, C1 = m1.values["A"], m1.values["B"], m1.values["C"]
eA1, eB1, eC1 = m1.errors["A"], m1.errors["B"], m1.errors["C"]
chi2_min1 = m1.fval
ndof1 = len(r) - 3

print("fit convergito:", m1.valid)
print(f"A = {A1:5.5g} ± {eA1:5.2g}")
print(f"B = {B1:5.4g} ± {eB1:5.2g}")
print(f"C = {C1:5.4g} ± {eC1:5.2g}")
print(f"chi2/ndof = {chi2_min1:5.4g}/{ndof1}")

print("\nB compatibile con 0? (errore statistico + sistematico)")
compatibile(B1, 0, np.sqrt(eB1**2 + sigma_sist**2))
print("C compatibile con 0?")
compatibile(C1, 0, eC1)

## Fit 2
Come il Fit 1 ma con l'esponente $D$ libero. Si parte dai risultati del Fit 1.

In [ ]:
def fitfunc2(r, A, B, C, D):
    return A / np.abs(r - B)**D + C

def chi2_2(A, B, C, D):
    sigma = np.sqrt(ey**2 + (ex * derivata(fitfunc2, r, A, B, C, D))**2)
    return np.sum(((y - fitfunc2(r, A, B, C, D)) / sigma)**2)

m2 = Minuit(chi2_2, A=A1, B=B1, C=C1, D=2.0)
m2.errordef = 1
m2.migrad()
m2.hesse()

A2, B2, C2, D2 = m2.values["A"], m2.values["B"], m2.values["C"], m2.values["D"]
eA2, eB2, eC2, eD2 = m2.errors["A"], m2.errors["B"], m2.errors["C"], m2.errors["D"]
chi2_min2 = m2.fval
ndof2 = len(r) - 4

print("fit convergito:", m2.valid)
print(f"A = {A2:5.5g} ± {eA2:5.2g}")
print(f"B = {B2:5.4g} ± {eB2:5.2g}")
print(f"C = {C2:5.4g} ± {eC2:5.2g}")
print(f"D = {D2:5.4g} ± {eD2:5.2g}")
print(f"chi2/ndof = {chi2_min2:5.4g}/{ndof2}")

print("\nD compatibile con 2?")
compatibile(D2, 2, eD2)

## Grafico 1: dati, fit e asintoti

In [ ]:
r_fit = np.linspace(r.min(), r.max(), 500)

plt.figure()
plt.errorbar(r, y, xerr=ex, yerr=ey, fmt=".", color="k", label="dati")
plt.plot(r_fit, fitfunc1(r_fit, A1, B1, C1), "r-",
         label=f"Fit 1: $\\chi^2$/ndof = {chi2_min1:5.4g}/{ndof1}\nA = {A1:5.5g} $\\pm$ {eA1:5.2g}\nB = {B1:5.4g} $\\pm$ {eB1:5.2g}\nC = {C1:5.4g} $\\pm$ {eC1:5.2g}")
plt.plot(r_fit, fitfunc2(r_fit, A2, B2, C2, D2), "m--", label=f"Fit 2 (D = {D2:5.3g})")
plt.axhline(C1, linestyle="--", color="b", label="asintoto orizzontale (C)")
plt.axvline(B1, linestyle="--", color="g", label="asintoto verticale (B)")

plt.title("Tensione in funzione della distanza")
plt.xlabel("r (mm)")
plt.ylabel("V (V)")
plt.legend()
plt.savefig("c1.png")
plt.show()

## Grafico 2: residui normalizzati
Se gli errori sono giusti dovrebbero seguire una gaussiana con media 0 e sigma 1.

In [ ]:
ey_eff = np.sqrt(ey**2 + (ex * derivata(fitfunc1, r, A1, B1, C1))**2)
res_norm = (y - fitfunc1(r, A1, B1, C1)) / ey_eff

fuori = np.sum(np.abs(res_norm) > 10)
if fuori > 0:
    print(f"ATTENZIONE: {fuori} punti hanno |residuo| > 10 e non compaiono nell'istogramma")

plt.figure()
counts, bins, _ = plt.hist(res_norm, range=(-10, 10), bins=20, alpha=0.6)
centri = (bins[:-1] + bins[1:]) / 2

def gauss(x, c, media, sigma):
    return c * np.exp(-0.5 * ((x - media) / sigma)**2)

# fit gaussiano solo sui bin non vuoti
mask = counts > 0
popt, pcov = curve_fit(gauss, centri[mask], counts[mask], p0=[counts.max(), 0.0, 1.0],
                       sigma=np.sqrt(counts[mask]), absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))
chi2_h = np.sum(((counts[mask] - gauss(centri[mask], *popt)) / np.sqrt(counts[mask]))**2)
ndof_h = np.count_nonzero(counts) - 3

xg = np.linspace(-10, 10, 400)
plt.plot(xg, gauss(xg, *popt), "r-",
         label=f"$\\chi^2$/ndof = {chi2_h:5.4g}/{ndof_h}\nCostante = {popt[0]:5.4g} $\\pm$ {perr[0]:5.2g}\nMedia = {popt[1]:5.4g} $\\pm$ {perr[1]:5.2g}\nSigma = {abs(popt[2]):5.4g} $\\pm$ {perr[2]:5.2g}")

plt.title("Residui")
plt.xlabel("Residui normalizzati")
plt.ylabel("Conteggi")
plt.legend()
plt.savefig("c2.png")
plt.show()

print("media compatibile con 0?")
compatibile(popt[1], 0, perr[1])
print("sigma compatibile con 1?")
compatibile(abs(popt[2]), 1, perr[2])

## Grafico 3: misura contro atteso
I punti devono stare sulla retta misura = atteso.

In [ ]:
atteso = fitfunc1(r, A1, B1, C1)

plt.figure()
plt.errorbar(atteso, y, yerr=ey_eff, fmt=".", color="k", label="misure")
plt.plot([atteso.min(), atteso.max()], [atteso.min(), atteso.max()], "r-", label="misura = atteso")

# se i punti si ammassano vicino a 0 togliere il # alle due righe sotto
# plt.xscale("log")
# plt.yscale("log")

plt.title("Compatibilità misure")
plt.xlabel("Atteso (V)")
plt.ylabel("Misura (V)")
plt.legend()
plt.savefig("c3.png")
plt.show()